# 02 - Feature Engineering
## Home Credit Default Risk

Pipeline completo de feature engineering incorporando os insights do EDA:
- `DAYS_EMPLOYED == 365243` é NA codificado → tratar
- `EXT_SOURCE_1/2/3` são as features mais preditivas → combinar
- Colunas do bloco imobiliário com >60% nulos → drop
- Agregações das tabelas auxiliares (bureau, previous, POS, installments, credit_card)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

from features import agg_numeric, agg_categorical

DATA_DIR = '../data'
PROC_DIR = '../data/processed'
os.makedirs(PROC_DIR, exist_ok=True)
print('Setup OK!')

## 1. Funções Auxiliares

In [ ]:
def reduce_memory(df, verbose=True):
    """Downcast de tipos numéricos para reduzir uso de memória."""
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    if verbose:
        print(f'  Memória: {start_mem:.1f}MB → {end_mem:.1f}MB ({100*(start_mem-end_mem)/start_mem:.1f}% redução)')
    return df

## 2. Carregamento e Feature Engineering da Tabela Principal

In [ ]:
def engineer_application_features(df):
    """Feature engineering na tabela principal com correções do EDA."""

    # ── Insight EDA: DAYS_EMPLOYED == 365243 é NA codificado ──
    # Clientes sem emprego registrado recebem esse valor absurdo
    df['FLAG_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(np.int8)
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    print(f'  DAYS_EMPLOYED anomalias tratadas: {df["FLAG_EMPLOYED_ANOMALY"].sum():,}')

    # ── Insight EDA: EXT_SOURCE_1 tem 56% de nulos mas é muito preditivo ──
    df['FLAG_EXT_SOURCE_1_MISSING'] = df['EXT_SOURCE_1'].isnull().astype(np.int8)

    # ── Features financeiras ──
    df['CREDIT_INCOME_RATIO']    = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_INCOME_RATIO']   = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
    df['ANNUITY_CREDIT_RATIO']   = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1)
    df['INCOME_PER_PERSON']      = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1)
    df['GOODS_CREDIT_RATIO']     = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1)

    # ── Features temporais / emprego ──
    df['DAYS_BIRTH_YEARS']          = (-df['DAYS_BIRTH'] / 365).astype(np.float32)  # idade em anos
    df['DAYS_EMPLOYED_YEARS']       = (-df['DAYS_EMPLOYED'] / 365).astype(np.float32)
    df['EMPLOYED_TO_BIRTH_RATIO']   = df['DAYS_EMPLOYED'] / (df['DAYS_BIRTH'] + 1)
    df['EMPLOYED_TO_AGE_PCT']       = df['DAYS_EMPLOYED_YEARS'] / (df['DAYS_BIRTH_YEARS'] + 1)

    # ── EXT_SOURCE combinadas (insight EDA: as 3 mais correlacionadas) ──
    ext = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']]
    df['EXT_SOURCE_MEAN']  = ext.mean(axis=1)
    df['EXT_SOURCE_STD']   = ext.std(axis=1).fillna(0)
    df['EXT_SOURCE_PROD']  = df['EXT_SOURCE_1'].fillna(0) * df['EXT_SOURCE_2'].fillna(0) * df['EXT_SOURCE_3'].fillna(0)
    df['EXT_SOURCE_MIN']   = ext.min(axis=1)
    # Interações par-a-par
    df['EXT12'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
    df['EXT13'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']
    df['EXT23'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

    # ── Contagem de documentos fornecidos ──
    doc_cols = [c for c in df.columns if 'FLAG_DOCUMENT' in c]
    df['DOCS_PROVIDED'] = df[doc_cols].sum(axis=1).astype(np.int8)

    # ── Drop de colunas com >60% nulos (bloco imobiliário sem valor preditivo) ──
    miss_pct = df.isnull().mean()
    cols_to_drop = miss_pct[miss_pct > 0.60].index.tolist()
    # Proteção: nunca dropamos as EXT_SOURCE mesmo que nulas
    cols_to_drop = [c for c in cols_to_drop if 'EXT_SOURCE' not in c and c != 'TARGET']
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    print(f'  Colunas dropadas (>60% nulos): {len(cols_to_drop)}')

    return df

print('Carregando application_train.csv...')
train = pd.read_csv(f'{DATA_DIR}/application_train.csv')
print('Carregando application_test.csv...')
test  = pd.read_csv(f'{DATA_DIR}/application_test.csv')

print(f'Shape original — Train: {train.shape} | Test: {test.shape}')

print('\nEngineering train...')
train = engineer_application_features(train)
print('Engineering test...')
test  = engineer_application_features(test)

train = reduce_memory(train)
test  = reduce_memory(test)
print(f'\nShape após engineering — Train: {train.shape} | Test: {test.shape}')

## 3. Agregação do Bureau (Histórico em Outros Bancos)

In [ ]:
print('Carregando bureau.csv...')
bureau = pd.read_csv(f'{DATA_DIR}/bureau.csv')
print('Carregando bureau_balance.csv...')
bureau_balance = pd.read_csv(f'{DATA_DIR}/bureau_balance.csv')

bureau = reduce_memory(bureau)
bureau_balance = reduce_memory(bureau_balance)

# Features manuais no bureau_balance: frequência de status bom (C/X) vs ruim (1-5)
bureau_balance['STATUS_GOOD'] = bureau_balance['STATUS'].isin(['C','X']).astype(np.int8)
bureau_balance['STATUS_LATE'] = bureau_balance['STATUS'].isin(['1','2','3','4','5']).astype(np.int8)

# Agrega bureau_balance → nível SK_ID_BUREAU
bb_agg_num = agg_numeric(bureau_balance, 'SK_ID_BUREAU', 'bb')
bb_agg_cat = agg_categorical(bureau_balance, 'SK_ID_BUREAU', 'bb')

# Merge com bureau
bureau = bureau.merge(bb_agg_num, on='SK_ID_BUREAU', how='left')
bureau = bureau.merge(bb_agg_cat, on='SK_ID_BUREAU', how='left')
del bureau_balance, bb_agg_num, bb_agg_cat; gc.collect()

# Features manuais no bureau
bureau['ACTIVE_CREDIT']  = (bureau['CREDIT_ACTIVE'] == 'Active').astype(np.int8)
bureau['CREDIT_OVERDUE'] = bureau['AMT_CREDIT_SUM_OVERDUE'].fillna(0).astype(np.float32)

# Agrega bureau → nível SK_ID_CURR
bureau_agg_num = agg_numeric(bureau, 'SK_ID_CURR', 'bureau')
bureau_agg_cat = agg_categorical(bureau, 'SK_ID_CURR', 'bureau')
del bureau; gc.collect()

print(f'Bureau agg num: {bureau_agg_num.shape} | cat: {bureau_agg_cat.shape}')

## 4. Tabelas Auxiliares (Previous Application + Históricos)

In [ ]:
tables = {
    'prev': 'previous_application.csv',
    'pos':  'POS_CASH_balance.csv',
    'ins':  'installments_payments.csv',
    'cc':   'credit_card_balance.csv'
}

aggregated = {}
for prefix, fname in tables.items():
    print(f'Processando {fname}...')
    df = pd.read_csv(f'{DATA_DIR}/{fname}')
    df = reduce_memory(df, verbose=False)

    # Features manuais por tabela
    if prefix == 'ins':
        # Diferença entre pago e devido — valores positivos = pagou mais (bom sinal)
        df['PAYMENT_DIFF']      = df['AMT_PAYMENT'] - df['AMT_INSTALMENT']
        df['PAYMENT_RATIO']     = df['AMT_PAYMENT'] / (df['AMT_INSTALMENT'] + 1)
        df['DAYS_PAST_DUE']     = (df['DAYS_ENTRY_PAYMENT'] - df['DAYS_INSTALMENT']).clip(lower=0)
        df['PAID_ON_TIME']      = (df['DAYS_PAST_DUE'] == 0).astype(np.int8)

    elif prefix == 'cc':
        # Utilização do cartão
        df['UTILIZATION'] = df['AMT_BALANCE'] / (df['AMT_CREDIT_LIMIT_ACTUAL'] + 1)

    elif prefix == 'prev':
        # Diferença entre o que foi pedido e o que foi aprovado
        df['APP_CREDIT_RATIO'] = df['AMT_APPLICATION'] / (df['AMT_CREDIT'] + 1)
        # Anos passados desde a aplicação anterior
        df['DAYS_DECISION_YEARS'] = -df['DAYS_DECISION'] / 365

    aggregated[f'{prefix}_num'] = agg_numeric(df, 'SK_ID_CURR', prefix)
    aggregated[f'{prefix}_cat'] = agg_categorical(df, 'SK_ID_CURR', prefix)
    del df; gc.collect()

print('\nTodas as tabelas processadas!')
for k, v in aggregated.items():
    print(f'  {k}: {v.shape}')

## 5. Merge Final

In [ ]:
def merge_all(df, bureau_agg_num, bureau_agg_cat, aggregated):
    df = df.merge(bureau_agg_num, on='SK_ID_CURR', how='left')
    df = df.merge(bureau_agg_cat, on='SK_ID_CURR', how='left')
    for key, frame in aggregated.items():
        df = df.merge(frame, on='SK_ID_CURR', how='left')
    return df

print('Fazendo merge final...')
train_full = merge_all(train, bureau_agg_num, bureau_agg_cat, aggregated)
test_full  = merge_all(test,  bureau_agg_num, bureau_agg_cat, aggregated)
del bureau_agg_num, bureau_agg_cat; gc.collect()

print(f'Train após merge: {train_full.shape}')
print(f'Test após merge:  {test_full.shape}')

## 6. One-Hot Encoding e Alinhamento

In [ ]:
# Drop de colunas com >80% nulos após o merge (tabelas auxiliares podem criar muitos)
miss_after = train_full.isnull().mean()
cols_high_miss = miss_after[miss_after > 0.80].index.tolist()
cols_high_miss = [c for c in cols_high_miss if c not in ['TARGET', 'SK_ID_CURR']]
print(f'Removendo {len(cols_high_miss)} colunas com >80% nulos pós-merge')
train_full.drop(columns=cols_high_miss, inplace=True, errors='ignore')
test_full.drop(columns=cols_high_miss, inplace=True, errors='ignore')

# One-Hot nas categorias de baixa cardinalidade
cat_cols = [c for c in train_full.select_dtypes('object').columns
            if train_full[c].nunique() < 10]
print(f'Aplicando One-Hot em {len(cat_cols)} colunas categóricas')

train_full = pd.get_dummies(train_full, columns=cat_cols)
test_full  = pd.get_dummies(test_full,  columns=cat_cols)

# Alinha colunas — test pode ter menos categorias após OHE
train_full, test_full = train_full.align(test_full, join='left', axis=1, fill_value=0)

# TARGET preenche com 0 no test (não existe lá)
if 'TARGET' in test_full.columns:
    test_full.drop(columns=['TARGET'], inplace=True)

# Sanitiza nomes de colunas — LightGBM não aceita caracteres especiais JSON ([], {}, :, etc.)
train_full.columns = train_full.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)
test_full.columns  = test_full.columns.str.replace(r'[^A-Za-z0-9_]', '_', regex=True)

print(f'\nShape final — Train: {train_full.shape} | Test: {test_full.shape}')
print(f'Colunas booleanas criadas pelo OHE: {[c for c in train_full.columns if train_full[c].dtype == bool][:3]}...')

## 7. Salvar Dataset Processado em Parquet

In [ ]:
# Converte booleans para int8 (parquet/lightgbm preferem)
bool_cols = train_full.select_dtypes(bool).columns
train_full[bool_cols] = train_full[bool_cols].astype(np.int8)
bool_cols_t = test_full.select_dtypes(bool).columns
test_full[bool_cols_t] = test_full[bool_cols_t].astype(np.int8)

train_full.to_parquet(f'{PROC_DIR}/train_processed.parquet', index=False)
test_full.to_parquet(f'{PROC_DIR}/test_processed.parquet', index=False)

print('Salvo em data/processed/')
print(f'  train_processed.parquet — {train_full.shape[1]} features, {len(train_full):,} linhas')
print(f'  test_processed.parquet  — {test_full.shape[1]} features, {len(test_full):,} linhas')

# Resumo rápido
import pandas as pd
feat_summary = pd.DataFrame({
    'dtype': train_full.dtypes,
    'null_pct': train_full.isnull().mean().round(3)
})
print(f'\nNull pct médio após processamento: {feat_summary.null_pct.mean():.2%}')
print(f'Tipos de dados:\n{feat_summary.dtype.value_counts().to_string()}')